# LightGBM urgency classification training

This notebook trains and evaluates three LightGBM-based urgency classification methods using the prepared dataset splits:
1. Multiclass urgency classification (`Low`, `Medium`, `High`, `Critical`)
2. Cost-sensitive multiclass urgency classification
3. Binary screening for `High`/`Critical` vs routine review

The notebook only prints classification reports and confusion matrices. It does not save models or metadata.

In [3]:
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from lightgbm import LGBMClassifier
except ImportError as exc:
    raise ImportError(
        "lightgbm is not installed. Install it in this environment before running the notebook."
    ) from exc

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_class_weight

cwd = Path.cwd().resolve()
candidate_roots = [cwd, *cwd.parents]
PROJECT_ROOT = None

for candidate_root in candidate_roots:
    if (candidate_root / "data" / "processed").exists():
        PROJECT_ROOT = candidate_root
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Could not locate project root from working directory: {cwd}"
    )

TRAIN_PATH = PROJECT_ROOT / "data" / "processed" / "incidents_train.csv"
VALIDATION_PATH = PROJECT_ROOT / "data" / "processed" / "incidents_validation.csv"
TEST_PATH = PROJECT_ROOT / "data" / "processed" / "incidents_test.csv"

RANDOM_SEED = 42
TARGET_RECALL = 0.95

print(f"Project root: {PROJECT_ROOT}")
print(f"Train path: {TRAIN_PATH}")
print(f"Validation path: {VALIDATION_PATH}")
print(f"Test path: {TEST_PATH}")


Project root: /home/lakshan/ssp/IMS
Train path: /home/lakshan/ssp/IMS/data/processed/incidents_train.csv
Validation path: /home/lakshan/ssp/IMS/data/processed/incidents_validation.csv
Test path: /home/lakshan/ssp/IMS/data/processed/incidents_test.csv


In [4]:
required_columns = {"feature_concatanation", "urgency", "urgency_label"}


def load_split(path):
    split_df = pd.read_csv(path)
    missing_columns = required_columns - set(split_df.columns)
    if missing_columns:
        raise ValueError(f"Dataset {path} is missing required columns: {sorted(missing_columns)}")

    split_df = split_df[["feature_concatanation", "urgency", "urgency_label"]].dropna().copy()
    split_df["feature_concatanation"] = split_df["feature_concatanation"].astype(str).str.strip()
    split_df = split_df[split_df["feature_concatanation"] != ""]
    return split_df


train_df = load_split(TRAIN_PATH)
validation_df = load_split(VALIDATION_PATH)
test_df = load_split(TEST_PATH)

label_mapping_df = (
    pd.concat(
        [
            train_df[["urgency", "urgency_label"]],
            validation_df[["urgency", "urgency_label"]],
            test_df[["urgency", "urgency_label"]],
        ],
        ignore_index=True,
    )
    .drop_duplicates()
    .sort_values("urgency_label")
    .reset_index(drop=True)
)
label_mapping = dict(zip(label_mapping_df["urgency_label"], label_mapping_df["urgency"]))

print(f"Train rows: {len(train_df)}")
print(f"Validation rows: {len(validation_df)}")
print(f"Test rows: {len(test_df)}")
label_mapping_df


Train rows: 2520
Validation rows: 540
Test rows: 540


,urgency,urgency_label
0,Low,0
1,Medium,1
2,High,2
3,Critical,3


In [5]:
def build_text_lightgbm_pipeline(**classifier_kwargs):
    return Pipeline(
        steps=[
            (
                "tfidf",
                TfidfVectorizer(
                    lowercase=True,
                    ngram_range=(1, 2),
                    min_df=2,
                    max_features=20000,
                    sublinear_tf=True,
                ),
            ),
            ("classifier", LGBMClassifier(**classifier_kwargs)),
        ]
    )


def print_multiclass_evaluation(title, y_true, y_pred, label_mapping):
    labels = sorted(label_mapping)
    print(title)
    print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
    print(f"Balanced accuracy: {balanced_accuracy_score(y_true, y_pred):.4f}")
    print(f"Macro F1: {f1_score(y_true, y_pred, average='macro'):.4f}")
    print("\nClassification report:\n")
    print(
        classification_report(
            y_true,
            y_pred,
            labels=labels,
            target_names=[label_mapping[label] for label in labels],
            zero_division=0,
        )
    )
    confusion_df = pd.DataFrame(
        confusion_matrix(y_true, y_pred, labels=labels),
        index=[f"actual_{label_mapping[label]}" for label in labels],
        columns=[f"pred_{label_mapping[label]}" for label in labels],
    )
    display(confusion_df)


def print_binary_evaluation(title, y_true, probabilities, threshold, label_mapping):
    predictions = (probabilities >= threshold).astype(int)
    print(title)
    print(f"Threshold: {threshold:.2f}")
    print(f"Accuracy: {accuracy_score(y_true, predictions):.4f}")
    print(f"Precision ({label_mapping[1]}): {precision_score(y_true, predictions, zero_division=0):.4f}")
    print(f"Recall ({label_mapping[1]}): {recall_score(y_true, predictions, zero_division=0):.4f}")
    print(f"F1 ({label_mapping[1]}): {f1_score(y_true, predictions, zero_division=0):.4f}")
    print(f"PR-AUC: {average_precision_score(y_true, probabilities):.4f}")
    print("\nClassification report:\n")
    print(
        classification_report(
            y_true,
            predictions,
            labels=[0, 1],
            target_names=[label_mapping[0], label_mapping[1]],
            zero_division=0,
        )
    )
    confusion_df = pd.DataFrame(
        confusion_matrix(y_true, predictions, labels=[0, 1]),
        index=[f"actual_{label_mapping[0]}", f"actual_{label_mapping[1]}"],
        columns=[f"pred_{label_mapping[0]}", f"pred_{label_mapping[1]}"],
    )
    display(confusion_df)


## 1. Multiclass LightGBM classifier

In [6]:
X_train = train_df["feature_concatanation"]
y_train = train_df["urgency_label"].astype(int)
X_validation = validation_df["feature_concatanation"]
y_validation = validation_df["urgency_label"].astype(int)
X_test = test_df["feature_concatanation"]
y_test = test_df["urgency_label"].astype(int)

baseline_urgency_classifier = build_text_lightgbm_pipeline(
    objective="multiclass",
    num_class=len(label_mapping),
    n_estimators=300,
    learning_rate=0.08,
    num_leaves=63,
    max_depth=-1,
    subsample=0.9,
    colsample_bytree=0.9,
    min_child_samples=20,
    reg_alpha=0.0,
    reg_lambda=0.0,
    random_state=RANDOM_SEED,
    n_jobs=-1,
    verbosity=-1,
)

baseline_urgency_classifier.fit(X_train, y_train)

baseline_validation_predictions = baseline_urgency_classifier.predict(X_validation)
baseline_test_predictions = baseline_urgency_classifier.predict(X_test)

print_multiclass_evaluation(
    "Validation results - baseline multiclass LightGBM",
    y_validation,
    baseline_validation_predictions,
    label_mapping,
)

print_multiclass_evaluation(
    "Test results - baseline multiclass LightGBM",
    y_test,
    baseline_test_predictions,
    label_mapping,
)


Validation results - baseline multiclass LightGBM
Accuracy: 0.5315
Balanced accuracy: 0.3530
Macro F1: 0.3567

Classification report:

              precision    recall  f1-score   support

         Low       0.00      0.00      0.00        10
      Medium       0.55      0.52      0.54       183
        High       0.54      0.66      0.59       258
    Critical       0.43      0.22      0.29        89

    accuracy                           0.53       540
   macro avg       0.38      0.35      0.36       540
weighted avg       0.51      0.53      0.51       540



/home/lakshan/ssp/IMS/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/lakshan/ssp/IMS/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,pred_Low,pred_Medium,pred_High,pred_Critical
actual_Low,0,8,2,0
actual_Medium,1,96,83,3
actual_High,1,62,171,24
actual_Critical,0,8,61,20


Test results - baseline multiclass LightGBM
Accuracy: 0.5926
Balanced accuracy: 0.3884
Macro F1: 0.3971

Classification report:

              precision    recall  f1-score   support

         Low       0.00      0.00      0.00         6
      Medium       0.62      0.55      0.58       192
        High       0.59      0.73      0.65       267
    Critical       0.48      0.28      0.35        75

    accuracy                           0.59       540
   macro avg       0.42      0.39      0.40       540
weighted avg       0.58      0.59      0.58       540



,pred_Low,pred_Medium,pred_High,pred_Critical
actual_Low,0,4,2,0
actual_Medium,0,105,82,5
actual_High,0,55,194,18
actual_Critical,0,4,50,21


## 2. Cost-sensitive multiclass LightGBM classifier

In [7]:
classes = np.array(sorted(label_mapping), dtype=int)
base_class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train,
)
class_weight_map = {label: float(weight) for label, weight in zip(classes, base_class_weights)}

severity_cost_multiplier = {
    label: 1.0 + (label / max(classes))
    for label in classes
}
weighted_class_map = {
    label: class_weight_map[label] * severity_cost_multiplier[label]
    for label in classes
}
sample_weights = y_train.map(weighted_class_map)

pd.DataFrame(
    {
        "urgency_label": classes,
        "urgency": [label_mapping[label] for label in classes],
        "class_weight": [class_weight_map[label] for label in classes],
        "severity_multiplier": [severity_cost_multiplier[label] for label in classes],
        "effective_weight": [weighted_class_map[label] for label in classes],
    }
)


,urgency_label,urgency,class_weight,severity_multiplier,effective_weight
0,0,Low,11.250000,1.000000,11.250000
1,1,Medium,0.798479,1.333333,1.064639
2,2,High,0.494118,1.666667,0.823529
3,3,Critical,1.575000,2.000000,3.150000


In [8]:
cost_sensitive_candidates = [
    {
        "model_name": "weighted_lgbm_v1",
        "n_estimators": 300,
        "learning_rate": 0.08,
        "num_leaves": 63,
        "min_child_samples": 20,
        "subsample": 0.9,
        "colsample_bytree": 0.9,
    },
    {
        "model_name": "weighted_lgbm_v2",
        "n_estimators": 500,
        "learning_rate": 0.05,
        "num_leaves": 127,
        "min_child_samples": 15,
        "subsample": 0.85,
        "colsample_bytree": 0.85,
    },
]

cost_sensitive_results = []
cost_sensitive_models = {}

for candidate in cost_sensitive_candidates:
    candidate_pipeline = build_text_lightgbm_pipeline(
        objective="multiclass",
        num_class=len(label_mapping),
        n_estimators=candidate["n_estimators"],
        learning_rate=candidate["learning_rate"],
        num_leaves=candidate["num_leaves"],
        min_child_samples=candidate["min_child_samples"],
        subsample=candidate["subsample"],
        colsample_bytree=candidate["colsample_bytree"],
        random_state=RANDOM_SEED,
        n_jobs=-1,
        verbosity=-1,
    )

    candidate_pipeline.fit(
        X_train,
        y_train,
        classifier__sample_weight=sample_weights.to_numpy(),
    )

    validation_predictions = candidate_pipeline.predict(X_validation)
    validation_accuracy = accuracy_score(y_validation, validation_predictions)
    validation_balanced_accuracy = balanced_accuracy_score(y_validation, validation_predictions)
    validation_macro_f1 = f1_score(y_validation, validation_predictions, average="macro")

    cost_sensitive_models[candidate["model_name"]] = candidate_pipeline
    cost_sensitive_results.append(
        {
            **candidate,
            "validation_accuracy": float(validation_accuracy),
            "validation_balanced_accuracy": float(validation_balanced_accuracy),
            "validation_macro_f1": float(validation_macro_f1),
        }
    )

cost_sensitive_results_df = pd.DataFrame(cost_sensitive_results).sort_values(
    by=["validation_balanced_accuracy", "validation_macro_f1", "validation_accuracy"],
    ascending=False,
).reset_index(drop=True)

best_cost_sensitive_model_name = cost_sensitive_results_df.iloc[0]["model_name"]
cost_sensitive_urgency_classifier = cost_sensitive_models[best_cost_sensitive_model_name]

cost_sensitive_results_df


/home/lakshan/ssp/IMS/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/lakshan/ssp/IMS/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,model_name,n_estimators,learning_rate,num_leaves,min_child_samples,subsample,colsample_bytree,validation_accuracy,validation_balanced_accuracy,validation_macro_f1
0,weighted_lgbm_v2,500,0.05,127,15,0.85,0.85,0.555556,0.380739,0.389390
1,weighted_lgbm_v1,300,0.08,63,20,0.90,0.90,0.524074,0.354920,0.360907


In [9]:
cost_validation_predictions = cost_sensitive_urgency_classifier.predict(X_validation)
cost_test_predictions = cost_sensitive_urgency_classifier.predict(X_test)

print(f"Selected cost-sensitive model: {best_cost_sensitive_model_name}")

print_multiclass_evaluation(
    "Validation results - cost-sensitive multiclass LightGBM",
    y_validation,
    cost_validation_predictions,
    label_mapping,
)

print_multiclass_evaluation(
    "Test results - cost-sensitive multiclass LightGBM",
    y_test,
    cost_test_predictions,
    label_mapping,
)


/home/lakshan/ssp/IMS/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/lakshan/ssp/IMS/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Selected cost-sensitive model: weighted_lgbm_v2
Validation results - cost-sensitive multiclass LightGBM
Accuracy: 0.5556
Balanced accuracy: 0.3807
Macro F1: 0.3894

Classification report:

              precision    recall  f1-score   support

         Low       0.00      0.00      0.00        10
      Medium       0.57      0.53      0.55       183
        High       0.56      0.68      0.61       258
    Critical       0.54      0.31      0.40        89

    accuracy                           0.56       540
   macro avg       0.42      0.38      0.39       540
weighted avg       0.55      0.56      0.54       540



,pred_Low,pred_Medium,pred_High,pred_Critical
actual_Low,0,8,2,0
actual_Medium,2,97,83,1
actual_High,1,59,175,23
actual_Critical,0,6,55,28


Test results - cost-sensitive multiclass LightGBM
Accuracy: 0.6037
Balanced accuracy: 0.3995
Macro F1: 0.4081

Classification report:

              precision    recall  f1-score   support

         Low       0.00      0.00      0.00         6
      Medium       0.64      0.56      0.60       192
        High       0.60      0.73      0.66       267
    Critical       0.48      0.31      0.37        75

    accuracy                           0.60       540
   macro avg       0.43      0.40      0.41       540
weighted avg       0.59      0.60      0.59       540



,pred_Low,pred_Medium,pred_High,pred_Critical
actual_Low,0,4,2,0
actual_Medium,0,107,79,6
actual_High,0,52,196,19
actual_Critical,0,4,48,23


## 3. Binary screening LightGBM classifier

In [10]:
positive_classes = {"High", "Critical"}

train_screen_df = train_df.copy()
validation_screen_df = validation_df.copy()
test_screen_df = test_df.copy()

for split_df in [train_screen_df, validation_screen_df, test_screen_df]:
    split_df["screening_label"] = split_df["urgency"].isin(positive_classes).astype(int)

X_train_screen = train_screen_df["feature_concatanation"]
y_train_screen = train_screen_df["screening_label"].astype(int)
X_validation_screen = validation_screen_df["feature_concatanation"]
y_validation_screen = validation_screen_df["screening_label"].astype(int)
X_test_screen = test_screen_df["feature_concatanation"]
y_test_screen = test_screen_df["screening_label"].astype(int)

screening_label_mapping = {0: "routine-review", 1: "immediate-attention"}

screening_classifier = build_text_lightgbm_pipeline(
    objective="binary",
    n_estimators=350,
    learning_rate=0.08,
    num_leaves=63,
    min_child_samples=20,
    subsample=0.9,
    colsample_bytree=0.9,
    class_weight="balanced",
    random_state=RANDOM_SEED,
    n_jobs=-1,
    verbosity=-1,
)

screening_classifier.fit(X_train_screen, y_train_screen)

validation_probabilities = screening_classifier.predict_proba(X_validation_screen)[:, 1]
test_probabilities = screening_classifier.predict_proba(X_test_screen)[:, 1]

threshold_candidates = np.round(np.arange(0.20, 0.61, 0.01), 2)
threshold_rows = []

for threshold in threshold_candidates:
    validation_predictions = (validation_probabilities >= threshold).astype(int)
    threshold_rows.append(
        {
            "threshold": float(threshold),
            "validation_precision": float(precision_score(y_validation_screen, validation_predictions, zero_division=0)),
            "validation_recall": float(recall_score(y_validation_screen, validation_predictions, zero_division=0)),
            "validation_f1": float(f1_score(y_validation_screen, validation_predictions, zero_division=0)),
            "validation_accuracy": float(accuracy_score(y_validation_screen, validation_predictions)),
        }
    )

threshold_report_df = pd.DataFrame(threshold_rows)

eligible_thresholds = threshold_report_df[threshold_report_df["validation_recall"] >= TARGET_RECALL]
if not eligible_thresholds.empty:
    selected_threshold_row = eligible_thresholds.sort_values(
        by=["validation_precision", "validation_f1", "threshold"],
        ascending=[False, False, True],
    ).iloc[0]
else:
    selected_threshold_row = threshold_report_df.sort_values(
        by=["validation_recall", "validation_precision", "validation_f1", "threshold"],
        ascending=[False, False, False, True],
    ).iloc[0]

selected_threshold = float(selected_threshold_row["threshold"])
threshold_report_df.sort_values("threshold").reset_index(drop=True).head()


/home/lakshan/ssp/IMS/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,threshold,validation_precision,validation_recall,validation_f1,validation_accuracy
0,0.20,0.752427,0.893372,0.816864,0.742593
1,0.21,0.755501,0.890490,0.817460,0.744444
2,0.22,0.755501,0.890490,0.817460,0.744444
3,0.23,0.755501,0.890490,0.817460,0.744444
4,0.24,0.754902,0.887608,0.815894,0.742593


In [11]:
print_binary_evaluation(
    "Validation results - binary screening LightGBM",
    y_validation_screen,
    validation_probabilities,
    selected_threshold,
    screening_label_mapping,
)

print_binary_evaluation(
    "Test results - binary screening LightGBM",
    y_test_screen,
    test_probabilities,
    selected_threshold,
    screening_label_mapping,
)


Validation results - binary screening LightGBM
Threshold: 0.20
Accuracy: 0.7426
Precision (immediate-attention): 0.7524
Recall (immediate-attention): 0.8934
F1 (immediate-attention): 0.8169
PR-AUC: 0.8597

Classification report:

                     precision    recall  f1-score   support

     routine-review       0.71      0.47      0.57       193
immediate-attention       0.75      0.89      0.82       347

           accuracy                           0.74       540
          macro avg       0.73      0.68      0.69       540
       weighted avg       0.74      0.74      0.73       540



,pred_routine-review,pred_immediate-attention
actual_routine-review,91,102
actual_immediate-attention,37,310


Test results - binary screening LightGBM
Threshold: 0.20
Accuracy: 0.7185
Precision (immediate-attention): 0.7262
Recall (immediate-attention): 0.8918
F1 (immediate-attention): 0.8005
PR-AUC: 0.8709

Classification report:

                     precision    recall  f1-score   support

     routine-review       0.69      0.42      0.52       198
immediate-attention       0.73      0.89      0.80       342

           accuracy                           0.72       540
          macro avg       0.71      0.66      0.66       540
       weighted avg       0.71      0.72      0.70       540



,pred_routine-review,pred_immediate-attention
actual_routine-review,83,115
actual_immediate-attention,37,305
